# Kaggle Home Depot — kalan deneyler tek notebook

Bu notebook Kaggle üstünde çalışacak şekilde standalone yazıldı. Local `src/` paketine ihtiyaç yok.

İçerik:
- `tfidf-svd-all-text + ridge` CV
- `ridge + tfidf-svd` HPO
- en iyi klasik modelden `submission.csv`
- opsiyonel `microsoft/deberta-v3-small` transformer deneyi

Not: mevcut bilinen en iyi Kaggle score: **0**. Notebook Kaggle output olarak `submission.csv` üretir; yükleme Kaggle arayüzünden yapılır. Türkçe karakter/Unicode temizliği `clean_text` içinde bozulmadan normalize edilir.

## Dependencies

Kaggle çoğunu hazır getirir (`numpy`, `pandas`, `scikit-learn`, `torch`). Eksik olursa bu hücre kurar. HPO için `optuna` gerekir. Transformer için `transformers`, `sentencepiece`, `accelerate` gerekir. Kaggle Internet kapalıysa bu paketleri notebook ortamına/dataset olarak ekle.

In [ ]:
# Dependency bootstrap for Kaggle
import importlib.util
import subprocess
import sys

INSTALL_MISSING = True
REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'optuna': 'optuna',
    # Needed only if RUN_TRANSFORMER=True later. Safe to keep installed.
    'transformers': 'transformers',
    'sentencepiece': 'sentencepiece',
    'accelerate': 'accelerate',
}

missing = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
print('Missing packages:', missing)
if missing and INSTALL_MISSING:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    print('Installed:', missing)
elif missing:
    raise RuntimeError(f'Missing packages: {missing}')
else:
    print('All required packages available.')

In [ ]:
import os, re, json, math, random, unicodedata, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RUN_CV_ALL_TEXT = True
RUN_HPO = True
RUN_SUBMISSION = True
RUN_TRANSFORMER = False  # GPU varsa True yap. Uzun sürer.

N_FOLDS = 5
HPO_TRIALS = 20
HPO_FOLDS = 3

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
print('Output dir:', OUTPUT_DIR.resolve())

## Data bul, yükle, merge et

In [ ]:
def find_file(names):
    roots = [Path('/kaggle/input'), Path('data'), Path('.')]
    for root in roots:
        if not root.exists():
            continue
        for name in names:
            direct = root / name
            if direct.exists():
                return direct
        for p in root.rglob('*'):
            if p.name in names:
                return p
    raise FileNotFoundError(f'Could not find any of: {names}')

train_path = find_file(['train.csv', 'train.csv.zip'])
test_path = find_file(['test.csv', 'test.csv.zip'])
desc_path = find_file(['product_descriptions.csv', 'product_descriptions.csv.zip'])
attr_path = find_file(['attributes.csv', 'attributes.csv.zip'])

print('train:', train_path)
print('test :', test_path)
print('desc :', desc_path)
print('attr :', attr_path)

train = pd.read_csv(train_path, encoding='ISO-8859-1')
test = pd.read_csv(test_path, encoding='ISO-8859-1')
desc = pd.read_csv(desc_path, encoding='ISO-8859-1')
attr = pd.read_csv(attr_path, encoding='ISO-8859-1')

print(train.shape, test.shape, desc.shape, attr.shape)
train.head()

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ''
    x = str(x)
    # Unicode normalize: Turkish chars like çğıöşü stay valid; weird forms collapse.
    x = unicodedata.normalize('NFKC', x)
    x = x.lower()
    x = x.replace('\u00a0', ' ')
    x = re.sub(r"(\d+)\s*['’]\s*", r'\1 ft ', x)
    x = re.sub(r'(\d+)\s*\"', r'\1 in ', x)
    x = re.sub(r'[^0-9a-zçğıöşü\s\.\-\+]', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x

def build_merged_dataset(train, test, desc, attr):
    train = train.copy()
    test = test.copy()
    desc = desc.copy()
    attr = attr.copy()

    for df in (train, test):
        df['search_term_raw'] = df['search_term'].map(clean_text)
        df['product_title_raw'] = df['product_title'].map(clean_text)

    desc['product_description'] = desc['product_description'].map(clean_text)
    train = train.merge(desc, on='product_uid', how='left')
    test = test.merge(desc, on='product_uid', how='left')
    train['product_description'] = train['product_description'].fillna('')
    test['product_description'] = test['product_description'].fillna('')

    attr = attr.dropna(subset=['product_uid']).copy()
    attr['name'] = attr['name'].fillna('').map(clean_text)
    attr['value'] = attr['value'].fillna('').map(clean_text)
    attr['attribute_text_raw'] = attr['name'] + ': ' + attr['value']
    attr_text = attr.groupby('product_uid')['attribute_text_raw'].agg(' | '.join).reset_index()

    train = train.merge(attr_text, on='product_uid', how='left')
    test = test.merge(attr_text, on='product_uid', how='left')
    train['attribute_text_raw'] = train['attribute_text_raw'].fillna('')
    test['attribute_text_raw'] = test['attribute_text_raw'].fillna('')

    for df in (train, test):
        df['product_text_raw'] = (
            df['product_title_raw'] + ' ' +
            df['product_description'] + ' ' +
            df['attribute_text_raw']
        ).str.strip()
        df['pair_text'] = (df['search_term_raw'] + ' [SEP] ' + df['product_text_raw']).str.strip()

    return train, test

train_df, test_df = build_merged_dataset(train, test, desc, attr)
print(train_df.shape, test_df.shape)
train_df[['id','search_term_raw','product_title_raw','relevance']].head()

## Yardımcı fonksiyonlar

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def relevance_bins(y):
    return pd.Series(y).round(2).astype(str)

def build_tfidf_svd_ridge(max_features=5000, svd_components=300, alpha=1.0, solver='auto', ngram_range=(1, 1)):
    return Pipeline([
        ('tfidf', TfidfVectorizer(max_features=max_features, sublinear_tf=True, ngram_range=ngram_range, stop_words='english')),
        ('svd', TruncatedSVD(n_components=svd_components, random_state=SEED)),
        ('ridge', Ridge(alpha=alpha, solver=solver, random_state=SEED)),
    ])

def run_text_cv(text_col, model, n_folds=N_FOLDS, name='experiment'):
    y = train_df['relevance'].values.astype(float)
    bins = relevance_bins(y)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    oof = np.zeros(len(train_df), dtype=float)
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, bins), 1):
        X_tr = train_df.iloc[tr_idx][text_col].fillna('')
        X_va = train_df.iloc[va_idx][text_col].fillna('')
        y_tr, y_va = y[tr_idx], y[va_idx]
        model.fit(X_tr, y_tr)
        pred = np.clip(model.predict(X_va), 1.0, 3.0)
        oof[va_idx] = pred
        score = rmse(y_va, pred)
        fold_rows.append({'fold': fold, 'rmse': score})
        print(f'{name} fold {fold}: RMSE={score:.6f}')

    full_score = rmse(y, oof)
    print(f'{name} OOF RMSE: {full_score:.6f}')
    oof_df = pd.DataFrame({'id': train_df['id'].values, 'relevance': y, 'prediction': oof})
    return full_score, pd.DataFrame(fold_rows), oof_df

## Deney 1 — `tfidf-svd-all-text + ridge` CV

In [ ]:
results = []
if RUN_CV_ALL_TEXT:
    all_text_model = build_tfidf_svd_ridge(max_features=5000, svd_components=300, alpha=1.0, solver='auto', ngram_range=(1, 1))
    score, folds, oof = run_text_cv('product_text_raw', all_text_model, N_FOLDS, 'tfidf-svd-all-text + ridge')
    results.append({'experiment': 'tfidf-svd-all-text + ridge', 'rmse': score})
    folds.to_csv(OUTPUT_DIR / 'tfidf_svd_all_text_ridge_folds.csv', index=False)
    oof.to_csv(OUTPUT_DIR / 'tfidf_svd_all_text_ridge_oof.csv', index=False)
    print(f'Saved OOF/folds to {OUTPUT_DIR}')

## Deney 2 — HPO `ridge + tfidf-svd`

In [ ]:
if RUN_HPO:
    try:
        import optuna
    except Exception as e:
        optuna = None
        print('Optuna yok. Kaggle Internet açıksa şu hücreyi çalıştırabilirsin: !pip install -q optuna')
        print('Hata:', repr(e))

    if optuna is not None:
        X_text = train_df['product_text_raw'].fillna('').values
        y = train_df['relevance'].values.astype(float)
        kf = KFold(n_splits=HPO_FOLDS, shuffle=True, random_state=SEED)

        def objective(trial):
            params = {
                'alpha': trial.suggest_float('alpha', 1e-3, 1e2, log=True),
                'solver': trial.suggest_categorical('solver', ['svd', 'lsqr', 'saga']),
                'max_features': trial.suggest_int('max_features', 1000, 20000, log=True),
                'ngram_range_max': trial.suggest_int('ngram_range_max', 1, 3),
                'svd_components': trial.suggest_int('svd_components', 50, 500, log=True),
            }
            scores = []
            for tr_idx, va_idx in kf.split(X_text):
                model = build_tfidf_svd_ridge(
                    max_features=params['max_features'],
                    svd_components=params['svd_components'],
                    alpha=params['alpha'],
                    solver=params['solver'],
                    ngram_range=(1, params['ngram_range_max']),
                )
                model.fit(X_text[tr_idx], y[tr_idx])
                pred = np.clip(model.predict(X_text[va_idx]), 1.0, 3.0)
                scores.append(rmse(y[va_idx], pred))
            return float(np.mean(scores))

        study = optuna.create_study(direction='minimize', study_name='hpo_ridge_tfidf')
        study.optimize(objective, n_trials=HPO_TRIALS, show_progress_bar=True)
        hpo_result = {
            'best_value': study.best_value,
            'best_params': study.best_params,
            'best_trial': study.best_trial.number,
        }
        print(json.dumps(hpo_result, indent=2))
        with open(OUTPUT_DIR / 'hpo_results.json', 'w') as f:
            json.dump(hpo_result, f, indent=2)
        results.append({'experiment': 'hpo ridge + tfidf-svd', 'rmse': study.best_value})

## Deney 3 — submission üret

Default: mevcut en iyi local CV yaklaşımı `tfidf-svd + ridge` ile `/kaggle/working/submission.csv`. HPO daha iyi çıkarsa `USE_HPO_FOR_SUBMISSION=True` yap.

In [ ]:
USE_HPO_FOR_SUBMISSION = False

if RUN_SUBMISSION:
    params = None
    if USE_HPO_FOR_SUBMISSION and (OUTPUT_DIR / 'hpo_results.json').exists():
        params = json.load(open(OUTPUT_DIR / 'hpo_results.json'))['best_params']

    if params:
        submission_model = build_tfidf_svd_ridge(
            max_features=params['max_features'],
            svd_components=params['svd_components'],
            alpha=params['alpha'],
            solver=params['solver'],
            ngram_range=(1, params['ngram_range_max']),
        )
        sub_name = 'submission_hpo_tfidf_svd_ridge.csv'
    else:
        submission_model = build_tfidf_svd_ridge(max_features=5000, svd_components=300, alpha=1.0, solver='auto', ngram_range=(1, 1))
        sub_name = 'submission.csv'

    submission_model.fit(train_df['product_text_raw'].fillna(''), train_df['relevance'].values.astype(float))
    preds = np.clip(submission_model.predict(test_df['product_text_raw'].fillna('')), 1.0, 3.0)
    submission = pd.DataFrame({'id': test_df['id'].values, 'relevance': preds})
    sub_path = OUTPUT_DIR / sub_name
    submission.to_csv(sub_path, index=False)
    print('Saved:', sub_path)
    print(submission.shape)
    print(submission['relevance'].describe())
    submission.head()

## Deney 4 — opsiyonel transformer/deberta

Uzun sürer. Kaggle GPU aç, `RUN_TRANSFORMER=True` yap. İnternet kapalıysa model dataset/cache olarak eklenmeli.

In [ ]:
if RUN_TRANSFORMER:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

    MODEL_NAME = 'microsoft/deberta-v3-small'
    MAX_LENGTH = 256
    BATCH_SIZE = 16
    EPOCHS = 3
    LR = 2e-5
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Device:', DEVICE)

    class RelevanceDataset(Dataset):
        def __init__(self, texts, targets, tokenizer, max_length):
            self.texts = list(texts)
            self.targets = None if targets is None else torch.tensor(targets, dtype=torch.float32)
            self.tokenizer = tokenizer
            self.max_length = max_length
        def __len__(self):
            return len(self.texts)
        def __getitem__(self, idx):
            enc = self.tokenizer(
                self.texts[idx], truncation=True, padding='max_length',
                max_length=self.max_length, return_tensors='pt'
            )
            item = {k: v.squeeze(0) for k, v in enc.items()}
            if self.targets is not None:
                item['labels'] = self.targets[idx]
            return item

    class TransformerRegressor(nn.Module):
        def __init__(self, model_name):
            super().__init__()
            self.encoder = AutoModel.from_pretrained(model_name)
            self.dropout = nn.Dropout(0.1)
            self.regressor = nn.Linear(self.encoder.config.hidden_size, 1)
        def forward(self, input_ids, attention_mask, labels=None, **kwargs):
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            pooled = out.last_hidden_state[:, 0]
            pred = self.regressor(self.dropout(pooled)).squeeze(-1)
            loss = None if labels is None else nn.MSELoss()(pred, labels)
            return pred, loss

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tr_df, va_df = train_test_split(train_df, test_size=0.2, random_state=SEED, stratify=relevance_bins(train_df['relevance']))
    tr_ds = RelevanceDataset(tr_df['pair_text'].fillna(''), tr_df['relevance'].values.astype(float), tokenizer, MAX_LENGTH)
    va_ds = RelevanceDataset(va_df['pair_text'].fillna(''), va_df['relevance'].values.astype(float), tokenizer, MAX_LENGTH)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)

    model = TransformerRegressor(MODEL_NAME).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    total_steps = len(tr_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    def eval_model():
        model.eval()
        preds, ys = [], []
        with torch.no_grad():
            for batch in va_loader:
                labels = batch.pop('labels').to(DEVICE)
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                pred, loss = model(**batch, labels=labels)
                preds.extend(pred.detach().cpu().numpy())
                ys.extend(labels.detach().cpu().numpy())
        return rmse(np.array(ys), np.clip(np.array(preds), 1.0, 3.0))

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for batch in tr_loader:
            labels = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            pred, loss = model(**batch, labels=labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            losses.append(loss.item())
        val_rmse = eval_model()
        print(f'Epoch {epoch}: train_loss={np.mean(losses):.6f} val_rmse={val_rmse:.6f}')

    torch.save(model.state_dict(), OUTPUT_DIR / 'deberta_v3_small_relevance.pt')
    print('Saved transformer weights')

## Özet

In [ ]:
summary = pd.DataFrame(results).sort_values('rmse') if results else pd.DataFrame()
if len(summary):
    summary.to_csv(OUTPUT_DIR / 'experiment_summary.csv', index=False)
summary